Objective: Predict House Sale Prices of Housing based on various features. Dataset Used Ames Housing data form Kaggle.com

In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.metrics import mean_squared_error,r2_score,root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import joblib



In [5]:
# Loading data into dataframe from kaggle AmesHousing.csv file downloaded
df_housing = pd.read_csv("AmesHousing.csv")
df_housing.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [6]:
# considering all  numeric columns for feture determination
df_hsg_no= df_housing.select_dtypes(include = [np.number] )

df_hsg_no.columns

Index(['Order', 'PID', 'MS SubClass', 'Lot Frontage', 'Lot Area',
       'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add',
       'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF',
       'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd',
       'Fireplaces', 'Garage Yr Blt', 'Garage Cars', 'Garage Area',
       'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch',
       'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold',
       'SalePrice'],
      dtype='object')

In [7]:
# removing low correlational columns
df_hsg_no_corr_desc = df_hsg_no.corr()['SalePrice'].sort_values(ascending = False)
df_hsg_no_corr_desc.drop('SalePrice',inplace = True)

low_correlation_matrix = df_hsg_no_corr_desc.abs() < df_hsg_no_corr_desc.abs().mean()
print(low_correlation_matrix)

# Filter the Series to keep only features with high correlation droping low correlation columns
low_correl_cols = df_hsg_no_corr_desc[low_correlation_matrix].index
print(low_correl_cols)

df_hsg_no.drop(columns = low_correl_cols,axis =1 , inplace = True)
df_hsg_no

Overall Qual       False
Gr Liv Area        False
Garage Cars        False
Garage Area        False
Total Bsmt SF      False
1st Flr SF         False
Year Built         False
Full Bath          False
Year Remod/Add     False
Garage Yr Blt      False
Mas Vnr Area       False
TotRms AbvGrd      False
Fireplaces         False
BsmtFin SF 1       False
Lot Frontage       False
Wood Deck SF       False
Open Porch SF      False
Half Bath           True
Bsmt Full Bath      True
2nd Flr SF          True
Lot Area            True
Bsmt Unf SF         True
Bedroom AbvGr       True
Screen Porch        True
Pool Area           True
Mo Sold             True
3Ssn Porch          True
BsmtFin SF 2        True
Misc Val            True
Yr Sold             True
Order               True
Bsmt Half Bath      True
Low Qual Fin SF     True
MS SubClass         True
Overall Cond        True
Kitchen AbvGr       True
Enclosed Porch      True
PID                 True
Name: SalePrice, dtype: bool
Index(['Half Bath', '

,Lot Frontage,Overall Qual,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,Total Bsmt SF,1st Flr SF,Gr Liv Area,Full Bath,TotRms AbvGrd,Fireplaces,Garage Yr Blt,Garage Cars,Garage Area,Wood Deck SF,Open Porch SF,SalePrice
0,141.0,6,1960,1960,112.0,639.0,1080.0,1656,1656,1,7,2,1960.0,2.0,528.0,210,62,215000
1,80.0,5,1961,1961,0.0,468.0,882.0,896,896,1,5,0,1961.0,1.0,730.0,140,0,105000
2,81.0,6,1958,1958,108.0,923.0,1329.0,1329,1329,1,6,0,1958.0,1.0,312.0,393,36,172000
3,93.0,7,1968,1968,0.0,1065.0,2110.0,2110,2110,2,8,2,1968.0,2.0,522.0,0,0,244000
4,74.0,5,1997,1998,0.0,791.0,928.0,928,1629,2,6,1,1997.0,2.0,482.0,212,34,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,37.0,6,1984,1984,0.0,819.0,1003.0,1003,1003,1,6,0,1984.0,2.0,588.0,120,0,142500
2926,NaN,5,1983,1983,0.0,301.0,864.0,902,902,1,5,0,1983.0,2.0,484.0,164,0,131000
2927,62.0,5,1992,1992,0.0,337.0,912.0,970,970,1,6,0,NaN,0.0,0.0,80,32,132000
2928,77.0,5,1974,1975,0.0,1071.0,1389.0,1389,1389,1,6,1,1975.0,2.0,418.0,240,38,170000


In [8]:
print("Columns remaining in df_hsg_no after dropping low correlation features:")
print(df_hsg_no.columns)

df_hsg_no.corr()['SalePrice'].sort_values(ascending=  False)

Columns remaining in df_hsg_no after dropping low correlation features:
Index(['Lot Frontage', 'Overall Qual', 'Year Built', 'Year Remod/Add',
       'Mas Vnr Area', 'BsmtFin SF 1', 'Total Bsmt SF', '1st Flr SF',
       'Gr Liv Area', 'Full Bath', 'TotRms AbvGrd', 'Fireplaces',
       'Garage Yr Blt', 'Garage Cars', 'Garage Area', 'Wood Deck SF',
       'Open Porch SF', 'SalePrice'],
      dtype='object')


,SalePrice
SalePrice,1.000000
Overall Qual,0.799262
Gr Liv Area,0.706780
Garage Cars,0.647877
Garage Area,0.640401
Total Bsmt SF,0.632280
1st Flr SF,0.621676
Year Built,0.558426
Full Bath,0.545604
Year Remod/Add,0.532974


In [9]:
df_hsg_no.shape
df_hsg_no.isnull().sum()


,0
Lot Frontage,490
Overall Qual,0
Year Built,0
Year Remod/Add,0
Mas Vnr Area,23
BsmtFin SF 1,1
Total Bsmt SF,1
1st Flr SF,0
Gr Liv Area,0
Full Bath,0


In [10]:
df_hsg_no.dropna(subset=['Lot Frontage','Garage Yr Blt','Mas Vnr Area'],inplace = True)
print('NUll Values present  in ''Lot Frontage,Garage Yr Blt,Mas Vnr Area'' columns so removved null rows')

# there are four columns containing null which are replaced by mean
df_hsg_no.fillna(np.round(df_hsg_no.mean()),inplace = True)

NUll Values present  in Lot Frontage,Garage Yr Blt,Mas Vnr Area columns so removved null rows


In [11]:
Correlation_matrix = df_hsg_no.corr()['SalePrice'].sort_values(ascending =  False)
Correlation_matrix.drop('SalePrice',inplace = True)
Correlation_matrix

,SalePrice
Overall Qual,0.803282
Gr Liv Area,0.713245
Garage Cars,0.661424
Garage Area,0.648142
Total Bsmt SF,0.643220
1st Flr SF,0.635926
Full Bath,0.560545
Year Built,0.559384
Garage Yr Blt,0.541557
Year Remod/Add,0.538719


In [12]:
# Trying different Methods for Feature selection
# 1. Using SelectKbest Method with F-regression which calculates f_value between fetures with target variable and finds relation

from sklearn.feature_selection import SelectKBest,f_regression
# UNivariant Feature Selection
X = df_hsg_no.drop('SalePrice',axis = 1)
Y = df_hsg_no['SalePrice']
selector = SelectKBest(score_func = f_regression,k= 10)
selector.fit(X,Y)
selected_final_features = X.columns[selector.get_support()]
print(selected_final_features)

Index(['Overall Qual', 'Year Built', 'Year Remod/Add', 'Total Bsmt SF',
       '1st Flr SF', 'Gr Liv Area', 'Full Bath', 'Garage Yr Blt',
       'Garage Cars', 'Garage Area'],
      dtype='object')


### Recursive Feature Elimination (RFE) with RandomForestRegressor

RFE works by recursively training an estimator and removing the least important features. We'll use RandomForestRegressor as our estimator, which provides feature importances, and select the top 10 features. In this Support_ gives the is the array of true and flase values which is obtained by RFE after coefiecient calculation and x.Columns(rfe_selector.support_) will identify only True columns from X Data frame


In [13]:
from sklearn.feature_selection import RFE

# Initialize RandomForestRegressor
estimator = RandomForestRegressor(n_estimators=100, random_state=42)

# Initialize RFE with the estimator and desired number of features
rfe_selector = RFE(estimator=estimator, n_features_to_select=10, step=1)

# Fit RFE to the data
rfe_selector.fit(X, Y)

# Get the selected features
rfe_selected_features = X.columns[rfe_selector.support_]

print("Selected features using RFE with RandomForestRegressor:")
print(rfe_selected_features)

Selected features using RFE with RandomForestRegressor:
Index(['Lot Frontage', 'Overall Qual', 'Year Built', 'Year Remod/Add',
       'BsmtFin SF 1', 'Total Bsmt SF', '1st Flr SF', 'Gr Liv Area',
       'Full Bath', 'Garage Area'],
      dtype='object')


### Using LassoCV for Optimal Alpha Selection

LassoCV is a version of Lasso that implements cross-validation to automatically choose the best alpha parameter. It fits the model for various alpha values and selects the alpha that yields the best performance. LassoCv finds th Optimal aplha value and trains the model(fit) with optimal value so no need of calculating alpha and then pass separately to lasso model

In [14]:
from sklearn.linear_model import LassoCV

lassocv_estimator = LassoCV(cv=5, random_state=42, n_jobs=-1) # n_jobs=-1 uses all available CPU cores

# Fit the LassoCV model to the data
lassocv_estimator.fit(X, Y)

# Get the optimal alpha found by cross-validation
optimal_alpha = lassocv_estimator.alpha_
print(f"Optimal alpha found by LassoCV: {optimal_alpha:.4f}")

# Get the coefficients with the optimal alpha
lassocv_coefficients = pd.Series(lassocv_estimator.coef_ , index=X.columns)

# Identify selected features (those with non-zero coefficients)
lassocv_selected_features = lassocv_coefficients[lassocv_coefficients != 0].index

print("\nSelected features using LassoCV (non-zero coefficients):")
print(lassocv_selected_features)

print("\nAll coefficients from LassoCV (with optimal alpha):")
print(lassocv_coefficients.sort_values(ascending=False))

Optimal alpha found by LassoCV: 56139.7719

Selected features using LassoCV (non-zero coefficients):
Index(['Year Built', 'Year Remod/Add', 'Mas Vnr Area', 'BsmtFin SF 1',
       'Total Bsmt SF', '1st Flr SF', 'Gr Liv Area', 'Garage Area',
       'Wood Deck SF'],
      dtype='object')

All coefficients from LassoCV (with optimal alpha):
Year Remod/Add    500.136162
Year Built        383.587358
Gr Liv Area        65.677512
Garage Area        63.973856
Mas Vnr Area       49.898845
Total Bsmt SF      29.297627
Wood Deck SF       26.919528
BsmtFin SF 1       18.149698
1st Flr SF          2.318489
Lot Frontage        0.000000
Overall Qual        0.000000
Full Bath           0.000000
TotRms AbvGrd      -0.000000
Garage Yr Blt       0.000000
Fireplaces          0.000000
Garage Cars         0.000000
Open Porch SF      -0.000000
dtype: float64


In [31]:
# Taking intersection of all features obtained from 3  methods above SelectKbest,RFE + RandomForestRegressor, Lassocv
common_features=(
    selected_final_features
    .intersection(lassocv_selected_features)
    .intersection(rfe_selected_features)
)
common_features = common_features.append(pd.Index(['Overall Qual']))
common_features

common_fetures_final = list(common_features) + ['SalePrice']
df_cleaned = df_hsg_no[common_fetures_final]
df_cleaned


,Year Built,Year Remod/Add,Total Bsmt SF,1st Flr SF,Gr Liv Area,Garage Area,Overall Qual,SalePrice
0,1960,1960,1080.0,1656,1656,528.0,6,215000
1,1961,1961,882.0,896,896,730.0,5,105000
2,1958,1958,1329.0,1329,1329,312.0,6,172000
3,1968,1968,2110.0,2110,2110,522.0,7,244000
4,1997,1998,928.0,928,1629,482.0,5,189900
...,...,...,...,...,...,...,...,...
2923,1977,1977,1126.0,1126,1126,484.0,5,160000
2924,1960,1996,1224.0,1224,1224,576.0,5,131000
2925,1984,1984,1003.0,1003,1003,588.0,6,142500
2928,1974,1975,1389.0,1389,1389,418.0,5,170000


In [32]:
df_cleaned.describe()

,Year Built,Year Remod/Add,Total Bsmt SF,1st Flr SF,Gr Liv Area,Garage Area,Overall Qual,SalePrice
count,2276.000000,2276.000000,2276.000000,2276.000000,2276.000000,2276.000000,2276.000000,2276.000000
mean,1971.982425,1984.878735,1065.846661,1165.786467,1504.092707,498.525483,6.179262,184383.589631
std,30.766510,21.161031,448.068024,399.153448,504.139663,193.830999,1.422714,83357.514485
min,1879.000000,1950.000000,0.000000,407.000000,407.000000,100.000000,1.000000,12789.000000
25%,1953.000000,1965.000000,793.000000,878.500000,1141.750000,350.000000,5.000000,130000.000000
50%,1974.000000,1994.000000,993.500000,1086.000000,1445.000000,483.000000,6.000000,161000.000000
75%,2003.000000,2005.000000,1324.000000,1392.000000,1734.000000,588.000000,7.000000,215000.000000
max,2010.000000,2010.000000,6110.000000,5095.000000,5642.000000,1488.000000,10.000000,755000.000000


In [43]:
# start Model Creation Spliting data
x = df_cleaned.drop('SalePrice',axis =1)
y = df_cleaned['SalePrice']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size =0.3,random_state =42)

In [49]:
# Model Creation

models= {
    'Linear Regression': LinearRegression(),
    'Lasso Regression': Lasso(alpha = 0.05),
    'RandomForest Regressor': RandomForestRegressor(random_state = 42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(random_state = 42),
    'xBoosting Regressor': XGBRegressor(objective = 'reg:squarederror', random_state = 42)

}

In [59]:
results = []
for model_name, model in models.items():
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    mse = mean_squared_error(y_test,y_pred)
    r2 = r2_score(y_test,y_pred)
    rmse = np.sqrt(mse)
    results.append((model_name,float(np.round(mse,4)),float(np.round(rmse,4)),float(np.round(r2,4))))
    print(f"{model_name} Results: MSE={float(np.round(mse,4))}, RMSE={float(np.round(rmse,4))}, R2={float(np.round(r2,4))}")


Linear Regression Results: MSE=1502874712.6709, RMSE=38766.9281, R2=0.7861
Lasso Regression Results: MSE=1502874819.7451, RMSE=38766.9295, R2=0.7861
RandomForest Regressor Results: MSE=886668643.1767, RMSE=29776.9818, R2=0.8738
Gradient Boosting Regressor Results: MSE=854719657.8593, RMSE=29235.5889, R2=0.8784
xBoosting Regressor Results: MSE=1003982528.0, RMSE=31685.6833, R2=0.8571


In [60]:
# Remove identified outliers from df_cleaned
df_cleaned_no_outliers = df_cleaned.drop(outliers.index)
print(f"Shape of df_cleaned before outlier removal: {df_cleaned.shape}")
print(f"Shape of df_cleaned after outlier removal: {df_cleaned_no_outliers.shape}")


Shape of df_cleaned before outlier removal: (2276, 8)
Shape of df_cleaned after outlier removal: (2240, 8)


In [61]:
df_cleaned_no_outliers.describe()

,Year Built,Year Remod/Add,Total Bsmt SF,1st Flr SF,Gr Liv Area,Garage Area,Overall Qual,SalePrice
count,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000
mean,1971.586161,1984.660714,1041.318750,1144.996429,1485.058036,493.083036,6.139286,180689.266071
std,30.762815,21.177846,394.825461,357.167339,471.273000,188.421795,1.388806,76222.331985
min,1879.000000,1950.000000,0.000000,407.000000,407.000000,100.000000,1.000000,12789.000000
25%,1952.000000,1965.000000,788.000000,874.000000,1136.000000,341.500000,5.000000,130000.000000
50%,1973.000000,1994.000000,988.000000,1078.000000,1436.000000,480.000000,6.000000,160000.000000
75%,2003.000000,2004.000000,1302.000000,1375.000000,1720.000000,578.000000,7.000000,213000.000000
max,2010.000000,2010.000000,2110.000000,3820.000000,3820.000000,1488.000000,10.000000,625000.000000


In [62]:
# start Model Creation Spliting data with cleaned data
x = df_cleaned_no_outliers.drop('SalePrice',axis =1)
y = df_cleaned_no_outliers['SalePrice']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size =0.3,random_state =42)


In [63]:
results = []
for model_name, model in models.items():
    model.fit(x_train,y_train)
    y_pred = model.predict(x_test)
    mse = mean_squared_error(y_test,y_pred)
    r2 = r2_score(y_test,y_pred)
    rmse = np.sqrt(mse)
    results.append((model_name,float(np.round(mse,4)),float(np.round(rmse,4)),float(np.round(r2,4))))
    print(f"{model_name} Results: MSE={float(np.round(mse,4))}, RMSE={float(np.round(rmse,4))}, R2={float(np.round(r2,4))}")


Linear Regression Results: MSE=965882581.8534, RMSE=31078.6515, R2=0.8448
Lasso Regression Results: MSE=965882708.4438, RMSE=31078.6536, R2=0.8448
RandomForest Regressor Results: MSE=674329661.5306, RMSE=25967.8582, R2=0.8916
Gradient Boosting Regressor Results: MSE=640970415.1462, RMSE=25317.3935, R2=0.897
xBoosting Regressor Results: MSE=698761344.0, RMSE=26434.0943, R2=0.8877


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the list of features to plot, excluding 'SalePrice'
features_to_plot = df_cleaned_no_outliers.drop(columns=['SalePrice']).columns

# Determine the number of rows and columns for the subplot grid
num_features = len(features_to_plot)
num_cols = 3  # You can adjust this number
num_rows = (num_features + num_cols - 1) // num_cols

# Create subplots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 5 * num_rows))
axes = axes.flatten() # Flatten the array of axes for easy iteration

for i, feature in enumerate(features_to_plot):
    sns.boxplot(y=df_cleaned_no_outliers[feature], ax=axes[i])
    axes[i].set_title(f'Box Plot of {feature}')
    axes[i].set_ylabel('') # Remove y-label to avoid clutter

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()
